# Bronze Pipeline — Resume After Map Failure

Runs the durable Map resume notebook, then continues with the normal
production Device and Registry notebooks. The defaults preserve the
interrupted full-refresh run while durable Map markers prevent replay.

In [0]:
def _ensure_resume_dropdown(name, default, choices, label):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


_ensure_resume_dropdown(
    "force_full_refresh", "true", ["true", "false"], "Force remaining refresh"
)
_ensure_resume_dropdown(
    "create_cutover_backups", "true", ["true", "false"], "Create cutover backups"
)
_ensure_resume_dropdown(
    "run_post_deployment_checks",
    "true",
    ["true", "false"],
    "Run post-deployment checks",
)
_ensure_resume_dropdown(
    "run_map_pipeline", "true", ["true", "false"], "Resume Map pipeline"
)
_ensure_resume_dropdown(
    "run_device_pipeline", "true", ["true", "false"], "Run Device pipeline"
)
_ensure_resume_dropdown(
    "run_registry_pipeline", "true", ["true", "false"], "Run Registry pipeline"
)

In [0]:
%run "/Workspace/Shared/ADC-DB/Prod/Pipelines/Bronze/_bronze_common"

In [0]:
_bronze_master_run_id = bronze_run_id()
_bronze_args = {
    "pipeline_run_id": _bronze_master_run_id,
    "force_full_refresh": str(
        bronze_bool("force_full_refresh", True)
    ).lower(),
    "create_cutover_backups": str(
        bronze_bool("create_cutover_backups", True)
    ).lower(),
    "run_post_deployment_checks": str(
        bronze_bool("run_post_deployment_checks", True)
    ).lower(),
}

_bronze_steps = [
    (
        "map_pipeline_resume",
        "/Workspace/Shared/PM-DE-Misc/Ben/Python/CC/New Bronze/map_pipeline",
        bronze_bool("run_map_pipeline", True),
    ),
    (
        "device_pipeline",
        "/Workspace/Shared/ADC-DB/Prod/Pipelines/Bronze/device_pipeline",
        bronze_bool("run_device_pipeline", True),
    ),
    (
        "registry_pipeline",
        "/Workspace/Shared/ADC-DB/Prod/Pipelines/Bronze/registry_pipeline",
        bronze_bool("run_registry_pipeline", True),
    ),
]

_bronze_results = []
_bronze_started_at = bronze_utc_now()

for _step_name, _step_path, _enabled in _bronze_steps:
    if not _enabled:
        _bronze_results.append({"pipeline": _step_name, "status": "SKIPPED"})
        continue

    print(
        f"[BRONZE RESUME] Starting {_step_name}; "
        f"master_run_id={_bronze_master_run_id}"
    )
    _step_args = dict(_bronze_args)
    if _step_name == "registry_pipeline":
        _step_args.update(
            {
                "target_schema": "4_prod.bronze",
                "expect_idempotent": "false",
            }
        )

    try:
        _raw_result = dbutils.notebook.run(_step_path, 0, _step_args)
        try:
            _step_result = json.loads(_raw_result)
        except Exception:
            _step_result = {
                "pipeline": _step_name,
                "status": "SUCCESS",
                "raw_result": _raw_result,
            }
        _bronze_results.append(_step_result)
        print(f"[BRONZE RESUME] Completed {_step_name}")
    except Exception as _step_error:
        _failure = {
            "pipeline": _step_name,
            "status": "FAILED",
            "error_type": type(_step_error).__name__,
            "error": str(_step_error),
        }
        _bronze_results.append(_failure)
        print(bronze_json(_failure))
        raise

_bronze_summary = {
    "status": "SUCCESS",
    "pipeline": "bronze_pipeline_resume",
    "run_id": _bronze_master_run_id,
    "started_at": _bronze_started_at,
    "completed_at": bronze_utc_now(),
    "force_full_refresh": bronze_bool("force_full_refresh", True),
    "execution_order": [step[0] for step in _bronze_steps if step[2]],
    "results": _bronze_results,
}
print(bronze_json(_bronze_summary))
dbutils.notebook.exit(bronze_json(_bronze_summary))